### Scaling Geospatial Workloads on Databricks Serverless Compute Using Grid Indexing + Delta Lake

### 🧭 Why Scaling Spatial Data Still Matters

Spatial data is big data.

Industries like fleets, retail, logistics, IoT, insurance, broadband, and utilities rely on millions to billions of location-based data points to answer questions such as:

- “Which drivers entered our congestion zone yesterday?”
- “How many customers live within 2 km of our store?”
- “Which telecom towers cover the densest user areas?”

These queries require mass spatial joins between points (e.g. GPS events) and polygons (zones, stores, service areas) — and naïvely doing this with full geometry math (ST_DISTANCE, ST_CONTAINS) is slow and expensive.

This is where grid indexing offers a fast and affordable solution — and works natively on Databricks Serverless Compute (even in the free version).

### What is Grid Indexing?
Grid indexing means dividing the map into square tiles and assigning every point to one of those tiles based on coordinates.
You can do this simply by rounding the latitude/longitude values — no external library required.

For example:

- FLOOR(lat * 100)  →  breaks latitude into ~1km cells
- FLOOR(lon * 100)  →  breaks longitude similarly

### Why this works:

- Converts expensive geometric spatial joins into string or integer equality joins
- Grid resolution can be tightened or relaxed (precision control)
- Fully supported in Databricks SQL, Python, and Delta Lake — including Serverless

Why Grid Indexing is Fast for Spatial Joins

1. Point in polygon -> Geometry math (ST_...) -> grid_id = grid_id string match
2. Distance filter -> Full ST_DISTANCE eval -> Use pre-filtered grids first
3. Performance -> Very slow at scale -> Scales using Spark + Photon

### Hands-On: Grid Indexing on Databricks

Let’s simulate spatial joins using grid cells with a practical example.

### Step 1: Create Sample Customer Locations with Grid IDs

In [0]:
from pyspark.sql.functions import expr
import random

# Generate 100 random lat/lon points around New York
data = [
    (
        f"Cust_{i}",
        40.70 + random.random() * 0.1,   # lat range near NYC
        -74.0 + random.random() * 0.1    # lon range near NYC
    )
    for i in range(100)
]

df_customers = spark.createDataFrame(data, ["customer_id", "lat", "lon"])

# Create geometry and grid cell (~1km precision)
df_customers = (
    df_customers
    .withColumn("geo_point", expr("ST_POINT(lon, lat)"))
    .withColumn("grid_cell", expr("concat(floor(lat * 100), '_', floor(lon * 100))"))
)

df_customers.createOrReplaceTempView("customers")
df_customers.show(5, truncate=False)

grid_cell is a fast, approximate spatial index.

In [0]:
store_data = [
    ("Store_Apple", 40.75, -73.99),
    ("Store_Samsung", 40.70, -73.95),
    ("Store_Oppo", 40.73, -74.03)
]

df_stores = spark.createDataFrame(store_data, ["store_id", "lat", "lon"])
df_stores = (
    df_stores
    .withColumn("geo_point", expr("ST_POINT(lon, lat)"))
    .withColumn("grid_cell", expr("concat(floor(lat * 100), '_', floor(lon * 100))"))
)

df_stores.createOrReplaceTempView("stores")
df_stores.show(truncate=False)

### Step 3: Spatial Join Using Grid Cells

In [0]:
%sql
SELECT
  c.customer_id,
  s.store_id,
  c.grid_cell
FROM customers c
JOIN stores s
  ON c.grid_cell = s.grid_cell;

This runs fast — using string equality join rather than geometry math.

### Exact Refinement for Accuracy

After the grid join, you can refine with ST_DISTANCE:

In [0]:
%sql
WITH filtered AS (
  SELECT
    c.customer_id,
    s.store_id,
    c.geo_point AS cust_loc,
    s.geo_point AS store_loc
  FROM customers c
  JOIN stores s
    ON c.grid_cell = s.grid_cell
)
SELECT
  customer_id,
  store_id,
  ST_DISTANCE(cust_loc, store_loc) AS distance_m
FROM filtered
WHERE ST_DISTANCE(cust_loc, store_loc) <= 5000; -- 5km cutoff